# Lab 5 – Word Embeddings and Clustering

This notebook follows the assignment description:
1. Train **Word2Vec**, **FastText**, and **Doc2Vec** models on the provided Hindi training data.
2. Perform **K‑means** clustering on the word vectors and list the 20 words closest to each centroid.
3. Represent each sentence in the validation (dev) and test sets as a vector using both the average word embeddings and the Doc2Vec inference vectors. For every sentence, find the most similar sentence (cosine similarity).

The code is heavily commented so that each step can be followed and reproduced.

In [3]:
# Imports
import pandas as pd, numpy as np, json, os, re
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Silence warnings for cleaner output
import warnings; warnings.filterwarnings('ignore')

In [5]:
# ---------------------------------------------------------------------
# 1. Load the data
# ---------------------------------------------------------------------
# Training files (different sizes) are located in `LAB 3/outputs/hin_Deva/`.
train_files = [
    '../LAB 3/outputs/hin_Deva/train_100000.csv',
    '../LAB 3/outputs/hin_Deva/train_300000.csv',
    '../LAB 3/outputs/hin_Deva/train_500000.csv'
]
train_sentences = []
for f in train_files:
    df = pd.read_csv(f)
    # The column containing raw text is named 'text'
    for txt in df['text'].astype(str):
        # simple_preprocess lower‑cases, removes punctuation and tokenises
        tokens = simple_preprocess(txt, deacc=True)
        if tokens:  # ignore empty lines
            train_sentences.append(tokens)

print(f'Loaded {len(train_sentences)} training sentences.')

Loaded 899532 training sentences.


In [7]:
# ---------------------------------------------------------------------
# 2. Train Word2Vec
# ---------------------------------------------------------------------
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=42
)
w2v_model.save('word2vec.model')
print('Word2Vec trained and saved.')

Word2Vec trained and saved.


In [8]:
# ---------------------------------------------------------------------
# 3. Train FastText
# ---------------------------------------------------------------------
ft_model = FastText(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=42
)
ft_model.save('fasttext.model')
print('FastText trained and saved.')

FastText trained and saved.


In [9]:
# ---------------------------------------------------------------------
# 4. Train Doc2Vec
# ---------------------------------------------------------------------
tagged_docs = [TaggedDocument(words=sent, tags=[str(i)]) for i, sent in enumerate(train_sentences)]
d2v_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=42
)
d2v_model.save('doc2vec.model')
print('Doc2Vec trained and saved.')

Doc2Vec trained and saved.


In [11]:
# ---------------------------------------------------------------------
# 5. K‑means clustering of Word2Vec word vectors
# ---------------------------------------------------------------------
# Extract the vocabulary and their vectors
words = list(w2v_model.wv.key_to_index.keys())
vectors = np.stack([w2v_model.wv[w] for w in words])
# Choose a reasonable number of clusters – 20 for illustration
n_clusters = 20
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
kmeans.fit(vectors)
centroids = kmeans.cluster_centers_
# For each centroid, find the 20 nearest words (cosine similarity)
cluster_words = {}
for idx in range(n_clusters):
    centroid = centroids[idx].reshape(1, -1)
    sims = cosine_similarity(centroid, vectors).flatten()
    top20_idx = sims.argsort()[-20:][::-1]
    cluster_words[idx] = [words[i] for i in top20_idx]

# Save cluster description to a JSON file for easy inspection
with open('word_clusters.json', 'w', encoding='utf8') as f:
    json.dump(cluster_words, f, ensure_ascii=False, indent=2)
print('Clusters saved to LAB 5/word_clusters.json')

Clusters saved to LAB 5/word_clusters.json


## 6. Sentence representations and nearest‑sentence search
* **Average Word2Vec** – mean of the word vectors that appear in the sentence.
* **Doc2Vec inference** – use the trained Doc2Vec model to infer a vector for a new sentence.

For each sentence in the validation (dev) and test sets we compute both vectors and then find the most similar sentence (cosine similarity) among the *combined* set of validation + test sentences.

In [13]:
# Load validation and test data
dev_path = '../LAB 3/outputs/hin_Deva/dev.csv'
test_path = '../LAB 3/outputs/hin_Deva/test.csv'
dev_df = pd.read_csv(dev_path)
test_df = pd.read_csv(test_path)
# Keep original text for later display
dev_texts = dev_df['text'].astype(str).tolist()
test_texts = test_df['text'].astype(str).tolist()

def preprocess(txt):
    return simple_preprocess(txt, deacc=True)

dev_tokens = [preprocess(t) for t in dev_texts]
test_tokens = [preprocess(t) for t in test_texts]

In [14]:
# Helper to compute average Word2Vec vector for a token list
def avg_w2v(tokens):
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if not vecs:
        return np.zeros(w2v_model.vector_size)
    return np.mean(vecs, axis=0)

In [15]:
# Compute vectors for all sentences
all_tokens = dev_tokens + test_tokens
avg_vectors = np.vstack([avg_w2v(t) for t in all_tokens])
# Doc2Vec inference – 20 epochs for a stable estimate
inf_vectors = np.vstack([d2v_model.infer_vector(t, epochs=20) for t in all_tokens])
# Concatenate both representations (optional – here we keep them separate)
combined_vectors = np.hstack([avg_vectors, inf_vectors])
print('Sentence vectors shape:', combined_vectors.shape)

Sentence vectors shape: (2000, 200)


In [16]:
# Find the most similar sentence for each entry (excluding itself)
similarities = cosine_similarity(combined_vectors)
np.fill_diagonal(similarities, -1)  # ignore self‑match
nearest_idx = similarities.argmax(axis=1)
# Build a DataFrame for easy viewing
all_texts = dev_texts + test_texts
results = pd.DataFrame({
    'sentence': all_texts,
    'nearest_sentence': [all_texts[i] for i in nearest_idx],
    'similarity': [similarities[i, nearest_idx[i]] for i in range(len(all_texts))]
})
# Show first 10 rows
results.head(10)

,sentence,nearest_sentence,similarity
0,"परीक्षाओं का टाल दिया जाना उसी क्रम में है, जि...",सरकार के रडार पर शिक्षा माफिया भी हैं।,0.789761
1,दून का मास्टर प्लान रद्द करने के अलावा नजूल भू...,सरकार के नियमों में परिवार के किसी और सदस्य को...,0.682990
2,समेकित राजस्व 648.3 करोड़ रुपये हो गया और इसमें...,सिगरेट श्रेणी में कंपनी का राजस्व 34 फीसदी बढ़...,0.831687
3,इटली और स्वीडन के छह प्रोफेसरों ने 32 दिनों के...,Bihar Politics सभी केंद्रीय विश्वविद्यालयों और...,0.620807
4,वहीं दीशू का कहना है कि उसकी बहन 11वीं कक्षा क...,लगा कि सरकारी स्कूल के बच्चे तो शायद ही इसका ज...,0.704296
5,खास बात यह है कि इस जांच में पुलिस की भी मदद ल...,जरूरत पर पुलिस उनका चालान भी काट सकती है।,0.787794
6,मौत की पुष्टि के बाद परिजन रोते बिलखते रहे और ...,पुलिस शव की पहचान करने का प्रयास कर रही है।,0.725000
7,उन लोगों ने खुद को स्वास्थ्य विभाग से होने की ...,"इसके साथ दुकान पर मास्क, सेनीटाइजर और कोरोना ग...",0.631626
8,"लिहाजा, आवेदनकर्ता द्वारा आवेदन भरने के बाद सं...",भारत में परीक्षा केंद्रों के लिए आवेदन कर रहे ...,0.614625
9,राऊ में खाती समाज के साढ़े छह फीसदी वोट हैं.,राज्य में मार्च और अप्रैल शादियों का मौसम माना...,0.673569


The notebook saves the trained models (`word2vec.model`, `fasttext.model`, `doc2vec.model`), the word‑cluster JSON file, and displays a table of each sentence with its most similar counterpart. Feel free to explore the clusters by loading `LAB 5/word_clusters.json`.